In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import importlib
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from plotly.subplots import make_subplots
from plotly import tools
import plotly.offline as pyo
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import glob
from scipy import stats
import uproot
from ROOT import TFile, TEfficiency, TH1D, TGraphAsymmErrors, RDataFrame, TCanvas

c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
ERROR in cling::CIFactory::createCI(): cannot extract standard library include paths!
Invoking:
  LC_ALL=C /Applications/Xcode.app/Contents/Developer/Toolchains/XcodeDefault.xctoolchain/usr/bin/c++ -isysroot;/Applications/Xcode.app/Contents/Developer/Platforms/MacOSX.platform/Developer/SDKs/MacOSX15.1.sdk   -xc++ -E -v /dev/null 2>&1 | sed -n -e '/^.include/,${' -e '/^ \/.*++/p' -e '}'
Results was:
c++: error: argument to '-isysroot' is missing (expected 1 value)
c++: error: no input files
With exit code 0


In [2]:
file = uproot.open("/Users/danielcarber/Documents/ICARUS/testtest.root")
print(file.keys())

['events;1', 'events/mc;1', 'events/mc/POT;3', 'events/mc/POT;2', 'events/mc/POT;1', 'events/mc/Livetime;3', 'events/mc/Livetime;2', 'events/mc/Livetime;1', 'events/mc/SelectedNu_Cuts;1', 'events/mc/SelectedCos_PhaseCuts;1', 'events/mc/Efficiency_PhaseCuts;9', 'events/mc/Efficiency_PhaseCuts;8']


In [5]:
Nu_Eff = file['events/mc/Efficiency_PhaseCuts;9']

Nu_Eff=Nu_Eff.arrays(library='pd')
print(Nu_Eff.keys())


Index(['all_1eNp_cut', 'category_topology', 'fiducial_cut', 'flash_cut',
       'nu_id', 'reco_electron_axial_spread', 'reco_electron_conv_dist',
       'reco_electron_dir_spread', 'reco_electron_energy',
       'reco_proton_muon_softmax', 'reco_proton_pion_softmax',
       'reco_proton_softmax', 'track_containment_cut', 'true_electrom_pT_mag',
       'true_electron_energy', 'true_proton_energy', 'Run', 'Subrun', 'Evt'],
      dtype='object')


In [7]:
quality_cuts = (Nu_Eff['true_electron_energy'] >0)
signal = Nu_Eff[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

efficiency = len(signal)/(len(Nu_Eff[quality_cuts]))
print(f"Purity of Directional Spread < 0.24: {efficiency*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Eff[quality_cuts])}")

Purity of Directional Spread < 0.24: 0.02%
Number of Signal and Total: 1492, 8576413


In [126]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Axial Spread > 0.02: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Axial Spread > 0.02: 80.82%
Number of Signal and Total: 1159, 1434


In [119]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02)&(Nu_Purity['reco_electron_dir_spread'] < 0.24)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Directional Spread < 0.24: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Directional Spread < 0.24: 83.76%
Number of Signal and Total: 1140, 1361


In [120]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02)&(Nu_Purity['reco_electron_dir_spread'] < 0.24)&(Nu_Purity['reco_electron_conv_dist'] <7.5)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Conversion Distance > 7.5: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Conversion Distance > 7.5: 85.13%
Number of Signal and Total: 1122, 1318


In [121]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton Softmax > 0.6: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton Softmax > 0.6: 86.76%
Number of Signal and Total: 1107, 1276


In [122]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton's Muon Softmax < 0,04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton's Muon Softmax < 0,04: 87.23%
Number of Signal and Total: 1107, 1269


In [123]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.24)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Proton's Pion Softmax < 0.24: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Proton's Pion Softmax < 0.24: 88.53%
Number of Signal and Total: 1050, 1186


In [124]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.24)&\
               (Nu_Purity['reco_electron_softmax'] <0.04)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Softmax < 0.04: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Softmax < 0.04: 91.19%
Number of Signal and Total: 611, 670


In [125]:
quality_cuts = (Nu_Purity['reco_electron_axial_spread'] > 0.02) & \
               (Nu_Purity['reco_electron_dir_spread'] < 0.24)&\
               (Nu_Purity['reco_electron_conv_dist'] <7.5)&\
               (Nu_Purity['reco_proton_softmax'] >0.6)&\
               (Nu_Purity['reco_proton_muon_softmax'] <0.04)&\
               (Nu_Purity['reco_proton_pion_softmax'] <0.24)&\
               (Nu_Purity['reco_electron_softmax'] <0.04)&\
               (Nu_Purity['reco_electron_primary_score'] >0.97)
signal = Nu_Purity[quality_cuts]
mask  = (signal['category_topology']==0) | (signal['category_topology']==1)
signal = signal[mask]

purity = len(signal)/(len(Nu_Purity[quality_cuts]))
print(f"Purity of Electron Primary Score > 0.97: {purity*100:.2f}%")
print(f"Number of Signal and Total: {len(signal)}, {len(Nu_Purity[quality_cuts])}")

Purity of Electron Primary Score > 0.97: 91.19%
Number of Signal and Total: 611, 670
